### Manual: 11-model

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Build the BEVFusion detection model from the resolved config.

Proves the config is self-consistent (camera 80ch + LiDAR 256ch -> ConvFuser 256,
decoder in 256, head in 512) and records parameter counts per submodule.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_model.py", log_name="step_model.log", timeout=1800)
report = sc.load_json("model.json")
print()
print("rc = %d" % rc)
print("params: %s" % json.dumps(report.get("model_params")))
print("init_cfg disabled (Swin pretrained download): %s" % report.get("init_cfg_disabled"))
print("init_cfg error: %s" % report.get("init_cfg_error"))
print("groups:")
for name in sorted(report.get("groups", {})):
    entry = report["groups"][name]
    print("  %-40s %-24s %12d" % (name, entry["type"], entry["params"]))
print("vtransform: %s" % json.dumps(report.get("vtransform"), sort_keys=True))
print("MODEL_CELL_OK" if "MODEL_OK" in text else "MODEL_CELL_FAILED (see logs/step_model.log)")
